# Ladder — evaluate the trained adapter

Scores `Ladder-3B` against the untouched base model by **running generated
programs against real Codeforces test cases**. No model-as-judge.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

This exists because the Kaggle GPU quota (6 h/week, not the 30 h advertised) ran
out after training. Colab has its own quota.

**Colab disconnects.** Results are written after every problem and a re-run picks
up where it stopped, so a dropped session costs minutes, not hours. If it drops,
just run the eval cells again.

## 1. Setup

Your Hugging Face token goes in Colab's secrets, not in this notebook:
**🔑 (left sidebar) → Add new secret → name `HF_TOKEN`** → paste a token with
read access → toggle *Notebook access* on.

The adapter lives in a private repo, so this step is required.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U bitsandbytes datasets
!git clone -q https://github.com/NiLabs-Models/ladder.git /content/ladder

In [ ]:
import sys
sys.path.insert(0, "/content/ladder/src")

from google.colab import userdata
from huggingface_hub import login, snapshot_download

login(userdata.get("HF_TOKEN"))

ADAPTER = snapshot_download(repo_id="ndemoss28/Ladder-3B", repo_type="model")
print("adapter:", ADAPTER)

## 2. Configure

40 problems at 4096 new tokens is the full run and takes roughly 3 h per model on
a T4. Colab free will probably not hold that in one session — which is fine,
because the eval resumes. Lower `NUM_PROBLEMS` if you would rather have a
coarser number sooner.

Be aware of the trade-off: fewer problems means a real improvement can hide in
the noise. At 20 problems, one solved problem is a 5-point swing in pass@1.

In [ ]:
from ladder.config import load_config

NUM_PROBLEMS = 40
MAX_NEW_TOKENS = 4096

cfg = load_config("/content/ladder/configs/ladder-3b-kaggle.yaml")
cfg.eval.num_problems = NUM_PROBLEMS
cfg.eval.max_new_tokens = MAX_NEW_TOKENS

# Results land in Drive if it is mounted, so a disconnect cannot lose them.
import os
OUT = "/content/drive/MyDrive/ladder" if os.path.isdir("/content/drive/MyDrive") else "/content/out"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)
print(f"{NUM_PROBLEMS} problems x {MAX_NEW_TOKENS} tokens, both models")

Optional but recommended — mount Drive first so results survive a disconnect:

```python
from google.colab import drive; drive.mount('/content/drive')
```

## 3. Score the base model

This is the number the fine-tune has to beat. Without it the tuned number means
nothing, so it runs first.

In [ ]:
import gc, torch
from ladder.eval.runner import evaluate
from ladder.eval.tasks import load_problems
from ladder.infer import load_for_inference, make_generator

problems = load_problems(cfg.eval, cfg.data)
print(f"{len(problems)} held-out problems")

def score(adapter, label):
    cfg.eval.results_path = f"{OUT}/eval-{label}.json"
    model, tok = load_for_inference(cfg, adapter)
    try:
        return evaluate(make_generator(model, tok, cfg), cfg, problems)
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

base = score(None, "base")
base["metrics"]

## 4. Score the fine-tune

Same problems, same prompt, same decoding. Only the adapter differs.

In [ ]:
tuned = score(ADAPTER, "tuned")
tuned["metrics"]

## 5. Results

In [ ]:
b = base["metrics"]["pass@1"]
t = tuned["metrics"]["pass@1"]

print(f"{'model':<34} {'pass@1':>8}")
print(f"{'Qwen2.5-Coder-3B-Instruct (base)':<34} {b:>8.3f}")
print(f"{'Ladder-3B':<34} {t:>8.3f}")
print(f"{'delta':<34} {t - b:>+8.3f}")
print()
print("problems     :", tuned["n_problems"])
print("base verdicts:", base["verdicts"])
print("tuned verdicts:", tuned["verdicts"])
print()
print("Send these two files back to record the run:")
print(" ", OUT + "/eval-base.json")
print(" ", OUT + "/eval-tuned.json")